# Notebook 4: Model Evaluation & Inference

**Site Revenue Prediction ML System - SageMaker Notebook Series**

This notebook covers:
- Loading trained model from checkpoint
- Detailed evaluation metrics (MAE, RMSE, SMAPE, R²)
- Prediction error analysis
- Actual vs Predicted visualizations
- Batch inference on new data
- SageMaker endpoint deployment preparation

## SageMaker Notes
- For inference endpoints, use `ml.m5.large` or `ml.c5.xlarge`
- Model artifacts should be packaged for SageMaker deployment
- Use SageMaker Batch Transform for large-scale batch inference

In [ ]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.path.abspath('')))

import torch
import torch.nn as nn
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import pickle

plt.style.use('seaborn-v0_8-whitegrid')

print(f"PyTorch version: {torch.__version__}")

## 1. Load Model and Preprocessor

In [ ]:
OUTPUT_DIR = Path('./outputs')

# Device selection
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f"Using device: {device}")

In [ ]:
# Define model architecture (same as training notebook)
class ResidualBlock(nn.Module):
    def __init__(self, in_features, out_features, dropout=0.2):
        super().__init__()
        self.linear1 = nn.Linear(in_features, out_features)
        self.bn1 = nn.BatchNorm1d(out_features)
        self.linear2 = nn.Linear(out_features, out_features)
        self.bn2 = nn.BatchNorm1d(out_features)
        self.dropout = nn.Dropout(dropout)
        self.projection = nn.Linear(in_features, out_features) if in_features != out_features else nn.Identity()
    
    def forward(self, x):
        residual = self.projection(x)
        out = torch.relu(self.bn1(self.linear1(x)))
        out = self.dropout(out)
        out = self.bn2(self.linear2(out))
        return torch.relu(out + residual)

class CategoricalEmbedding(nn.Module):
    def __init__(self, vocab_sizes, embedding_dim=16):
        super().__init__()
        self.embeddings = nn.ModuleDict()
        self.feature_names = list(vocab_sizes.keys())
        for name, vocab_size in vocab_sizes.items():
            dim = min(embedding_dim, max(4, (vocab_size + 1) // 2))
            self.embeddings[name] = nn.Embedding(vocab_size + 1, dim, padding_idx=0)
        self.output_dim = sum(self.embeddings[name].embedding_dim for name in self.feature_names)
    
    def forward(self, x):
        embeddings = []
        for i, name in enumerate(self.feature_names):
            idx = x[:, i].clamp(0, self.embeddings[name].num_embeddings - 1)
            embeddings.append(self.embeddings[name](idx))
        return torch.cat(embeddings, dim=1)

class SiteScoringModel(nn.Module):
    def __init__(self, n_numeric, n_boolean, categorical_vocab_sizes, embedding_dim=16, hidden_dims=None, dropout=0.2):
        super().__init__()
        hidden_dims = hidden_dims or [512, 256, 128, 64]
        self.cat_embedding = CategoricalEmbedding(categorical_vocab_sizes, embedding_dim)
        self.numeric_bn = nn.BatchNorm1d(n_numeric) if n_numeric > 0 else None
        self.n_numeric = n_numeric
        self.n_boolean = n_boolean
        total_input_dim = self.cat_embedding.output_dim + n_numeric + n_boolean
        layers = []
        in_dim = total_input_dim
        for out_dim in hidden_dims:
            layers.append(ResidualBlock(in_dim, out_dim, dropout))
            in_dim = out_dim
        self.mlp = nn.Sequential(*layers)
        self.output = nn.Linear(hidden_dims[-1], 1)
    
    def forward(self, numeric, categorical, boolean):
        cat_embedded = self.cat_embedding(categorical)
        if self.numeric_bn is not None and self.n_numeric > 0:
            numeric = self.numeric_bn(numeric)
        x = torch.cat([cat_embedded, numeric, boolean], dim=1)
        x = self.mlp(x)
        return self.output(x)

print("Model classes defined")

In [ ]:
# Load checkpoint
checkpoint = torch.load(OUTPUT_DIR / 'best_model.pt', map_location=device, weights_only=False)
config = checkpoint['config']

print("Loaded model configuration:")
for key, value in config.items():
    if key != 'categorical_vocab_sizes':
        print(f"  {key}: {value}")

# Recreate model
model = SiteScoringModel(
    n_numeric=config['n_numeric'],
    n_boolean=config['n_boolean'],
    categorical_vocab_sizes=config['categorical_vocab_sizes'],
    embedding_dim=config['embedding_dim'],
    hidden_dims=config['hidden_dims'],
    dropout=config['dropout'],
)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

print(f"\nModel loaded successfully!")
print(f"Best validation loss: {checkpoint['best_val_loss']:.4f}")

In [ ]:
# Load preprocessor
with open(OUTPUT_DIR / 'preprocessor.pkl', 'rb') as f:
    preprocessor = pickle.load(f)

target_scaler = preprocessor['target_scaler']
feature_scaler = preprocessor['scaler']
label_encoders = preprocessor['label_encoders']

print(f"Preprocessor loaded with {len(label_encoders)} categorical encoders")

## 2. Load Test Data

In [ ]:
# Load processed data
data = torch.load(OUTPUT_DIR / 'processed_data.pt', weights_only=False)

numeric = data['numeric']
categorical = data['categorical']
boolean = data['boolean']
target = data['target']
test_idx = data['test_idx']

# Get test set
test_numeric = numeric[test_idx]
test_categorical = categorical[test_idx]
test_boolean = boolean[test_idx]
test_target = target[test_idx]

print(f"Test set size: {len(test_idx):,}")

## 3. Generate Predictions

In [ ]:
@torch.no_grad()
def predict_batch(model, numeric, categorical, boolean, device, batch_size=4096):
    """Generate predictions in batches."""
    model.eval()
    predictions = []
    
    n_samples = len(numeric)
    for i in range(0, n_samples, batch_size):
        end_idx = min(i + batch_size, n_samples)
        batch_numeric = numeric[i:end_idx].to(device)
        batch_categorical = categorical[i:end_idx].to(device)
        batch_boolean = boolean[i:end_idx].to(device)
        
        batch_pred = model(batch_numeric, batch_categorical, batch_boolean)
        predictions.append(batch_pred.cpu())
    
    return torch.cat(predictions)

# Generate predictions
predictions_scaled = predict_batch(model, test_numeric, test_categorical, test_boolean, device)
print(f"Generated {len(predictions_scaled):,} predictions")

In [ ]:
# Convert to original scale
predictions_np = predictions_scaled.numpy()
targets_np = test_target.numpy()

if target_scaler is not None:
    predictions_orig = target_scaler.inverse_transform(predictions_np).flatten()
    targets_orig = target_scaler.inverse_transform(targets_np).flatten()
else:
    predictions_orig = predictions_np.flatten()
    targets_orig = targets_np.flatten()

print(f"Predictions range: ${predictions_orig.min():,.0f} - ${predictions_orig.max():,.0f}")
print(f"Targets range: ${targets_orig.min():,.0f} - ${targets_orig.max():,.0f}")

## 4. Comprehensive Metrics

In [ ]:
def calculate_metrics(y_true, y_pred):
    """Calculate comprehensive regression metrics."""
    # Mean Absolute Error
    mae = np.mean(np.abs(y_true - y_pred))
    
    # Root Mean Square Error
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    
    # Mean Absolute Percentage Error (avoid division by zero)
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    
    # Symmetric MAPE (handles zero values better)
    smape = np.mean(2 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred) + 1e-8)) * 100
    
    # R² Score
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
    
    # Median Absolute Error (robust to outliers)
    median_ae = np.median(np.abs(y_true - y_pred))
    
    return {
        'MAE': mae,
        'RMSE': rmse,
        'MAPE': mape,
        'SMAPE': smape,
        'R²': r2,
        'Median_AE': median_ae,
    }

metrics = calculate_metrics(targets_orig, predictions_orig)

print("Test Set Metrics:")
print("=" * 50)
print(f"  MAE (Mean Absolute Error):     ${metrics['MAE']:,.2f}")
print(f"  RMSE (Root Mean Square Error): ${metrics['RMSE']:,.2f}")
print(f"  Median AE (Median Abs Error):  ${metrics['Median_AE']:,.2f}")
print(f"  MAPE (Mean Abs % Error):       {metrics['MAPE']:.2f}%")
print(f"  SMAPE (Symmetric MAPE):        {metrics['SMAPE']:.2f}%")
print(f"  R² Score:                      {metrics['R²']:.4f}")

## 5. Visualization: Actual vs Predicted

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Scatter plot: Actual vs Predicted
ax = axes[0, 0]
ax.scatter(targets_orig, predictions_orig, alpha=0.3, s=10)
max_val = max(targets_orig.max(), predictions_orig.max())
ax.plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Perfect Prediction')
ax.set_xlabel('Actual Revenue ($)')
ax.set_ylabel('Predicted Revenue ($)')
ax.set_title(f'Actual vs Predicted (R² = {metrics["R²"]:.4f})')
ax.legend()
ax.grid(True, alpha=0.3)

# Residual distribution
ax = axes[0, 1]
residuals = targets_orig - predictions_orig
ax.hist(residuals, bins=50, edgecolor='white', alpha=0.7)
ax.axvline(0, color='red', linestyle='--', linewidth=2)
ax.axvline(np.mean(residuals), color='green', linestyle='--', linewidth=2, label=f'Mean: ${np.mean(residuals):,.0f}')
ax.set_xlabel('Prediction Error ($)')
ax.set_ylabel('Count')
ax.set_title('Residual Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

# Residuals vs Predicted (check for heteroscedasticity)
ax = axes[1, 0]
ax.scatter(predictions_orig, residuals, alpha=0.3, s=10)
ax.axhline(0, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Predicted Revenue ($)')
ax.set_ylabel('Residual ($)')
ax.set_title('Residuals vs Predicted (Check for Patterns)')
ax.grid(True, alpha=0.3)

# Error by revenue range
ax = axes[1, 1]
percentiles = [0, 25, 50, 75, 90, 100]
bins = np.percentile(targets_orig, percentiles)
bin_labels = [f'${bins[i]:,.0f}-${bins[i+1]:,.0f}' for i in range(len(bins)-1)]
bin_indices = np.digitize(targets_orig, bins[1:-1])

mae_by_bin = []
for i in range(len(bins)-1):
    mask = bin_indices == i
    if mask.sum() > 0:
        mae_by_bin.append(np.mean(np.abs(residuals[mask])))
    else:
        mae_by_bin.append(0)

ax.bar(range(len(mae_by_bin)), mae_by_bin, tick_label=bin_labels)
ax.set_xlabel('Revenue Range')
ax.set_ylabel('MAE ($)')
ax.set_title('MAE by Revenue Range')
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'evaluation_plots.png', dpi=150)
plt.show()

print(f"Evaluation plots saved to {OUTPUT_DIR / 'evaluation_plots.png'}")

## 6. Error Analysis by Feature Groups

In [ ]:
# Analyze errors by prediction confidence
abs_errors = np.abs(residuals)
pct_errors = np.abs(residuals / (targets_orig + 1)) * 100

print("Error Statistics:")
print("=" * 50)
print(f"  Mean absolute error:   ${np.mean(abs_errors):,.2f}")
print(f"  Median absolute error: ${np.median(abs_errors):,.2f}")
print(f"  90th percentile error: ${np.percentile(abs_errors, 90):,.2f}")
print(f"  95th percentile error: ${np.percentile(abs_errors, 95):,.2f}")
print(f"  Max error:             ${np.max(abs_errors):,.2f}")

print(f"\nPercentage Error Statistics:")
print(f"  Mean % error:   {np.mean(pct_errors):.1f}%")
print(f"  Median % error: {np.median(pct_errors):.1f}%")

# Sites with good predictions (< 20% error)
good_pred_mask = pct_errors < 20
print(f"\nPrediction Accuracy:")
print(f"  Sites with <10% error: {(pct_errors < 10).sum():,} ({(pct_errors < 10).mean()*100:.1f}%)")
print(f"  Sites with <20% error: {(pct_errors < 20).sum():,} ({(pct_errors < 20).mean()*100:.1f}%)")
print(f"  Sites with <30% error: {(pct_errors < 30).sum():,} ({(pct_errors < 30).mean()*100:.1f}%)")

## 7. SiteScorer Class for Inference

In [ ]:
class SiteScorer:
    """
    Production-ready inference class.
    Loads model and preprocessor, handles new data predictions.
    """
    
    def __init__(self, model_path, preprocessor_path, device=None):
        # Device selection
        if device is None:
            if torch.cuda.is_available():
                device = 'cuda'
            elif torch.backends.mps.is_available():
                device = 'mps'
            else:
                device = 'cpu'
        self.device = torch.device(device)
        
        # Load checkpoint
        checkpoint = torch.load(model_path, map_location=self.device, weights_only=False)
        config = checkpoint['config']
        
        # Load preprocessor
        with open(preprocessor_path, 'rb') as f:
            self.preprocessor = pickle.load(f)
        
        # Recreate model
        self.model = SiteScoringModel(
            n_numeric=config['n_numeric'],
            n_boolean=config['n_boolean'],
            categorical_vocab_sizes=config['categorical_vocab_sizes'],
            embedding_dim=config['embedding_dim'],
            hidden_dims=config['hidden_dims'],
            dropout=config['dropout'],
        )
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.model.to(self.device)
        self.model.eval()
        
        self.target_scaler = self.preprocessor['target_scaler']
        self.feature_scaler = self.preprocessor['scaler']
        self.label_encoders = self.preprocessor['label_encoders']
        self.available_numeric = self.preprocessor.get('available_numeric', [])
        self.available_categorical = self.preprocessor.get('available_categorical', [])
        self.available_boolean = self.preprocessor.get('available_boolean', [])
    
    @torch.no_grad()
    def predict(self, df):
        """Make predictions on a Polars/Pandas DataFrame."""
        # Convert to Polars if needed
        if isinstance(df, pd.DataFrame):
            df = pl.from_pandas(df)
        
        # Process features
        numeric = self._process_numeric(df)
        categorical = self._process_categorical(df)
        boolean = self._process_boolean(df)
        
        # Move to device
        numeric = torch.from_numpy(numeric).to(self.device)
        categorical = torch.from_numpy(categorical).to(self.device)
        boolean = torch.from_numpy(boolean).to(self.device)
        
        # Predict
        predictions = self.model(numeric, categorical, boolean)
        
        # Inverse transform
        predictions_np = predictions.cpu().numpy()
        if self.target_scaler is not None:
            return self.target_scaler.inverse_transform(predictions_np).flatten()
        return predictions_np.flatten()
    
    def _process_numeric(self, df):
        available = [c for c in self.available_numeric if c in df.columns]
        numeric_df = df.select(available).fill_null(0).fill_nan(0)
        numeric_array = numeric_df.to_numpy().astype(np.float32)
        return self.feature_scaler.transform(numeric_array).astype(np.float32)
    
    def _process_categorical(self, df):
        available = [c for c in self.available_categorical if c in df.columns]
        encoded_cols = []
        for col in available:
            col_data = df.select(col).fill_null("__MISSING__").to_series().to_list()
            le = self.label_encoders.get(col)
            if le is not None:
                encoded = []
                for val in col_data:
                    if val in le.classes_:
                        encoded.append(le.transform([val])[0])
                    else:
                        encoded.append(0)
                encoded_cols.append(encoded)
            else:
                encoded_cols.append([0] * len(df))
        return np.column_stack(encoded_cols).astype(np.int64)
    
    def _process_boolean(self, df):
        available = [c for c in self.available_boolean if c in df.columns]
        bool_cols = []
        for col in available:
            col_data = df.select(col).to_series()
            if col_data.dtype == pl.Boolean:
                values = col_data.fill_null(False).to_numpy().astype(np.float32)
            elif col_data.dtype in [pl.Int8, pl.Int16, pl.Int32, pl.Int64]:
                values = col_data.fill_null(0).to_numpy().astype(np.float32)
            else:
                values = col_data.fill_null("false").str.to_lowercase().is_in(["true", "1", "yes"]).to_numpy().astype(np.float32)
            bool_cols.append(values)
        if bool_cols:
            return np.column_stack(bool_cols).astype(np.float32)
        return np.zeros((len(df), 1), dtype=np.float32)

print("SiteScorer class defined")

In [ ]:
# Test SiteScorer
scorer = SiteScorer(
    model_path=OUTPUT_DIR / 'best_model.pt',
    preprocessor_path=OUTPUT_DIR / 'preprocessor.pkl',
)

# Load original data for testing
original_df = pl.read_parquet('../data/processed/site_training_data.parquet')
sample_df = original_df.head(100)

# Make predictions
sample_predictions = scorer.predict(sample_df)

print(f"Sample predictions (first 10):")
for i in range(10):
    actual = sample_df['avg_monthly_revenue'][i]
    predicted = sample_predictions[i]
    print(f"  Site {i+1}: Actual=${actual:,.0f}, Predicted=${predicted:,.0f}")

## 8. Save Evaluation Results

In [ ]:
# Save comprehensive evaluation results
evaluation_results = {
    'metrics': {k: float(v) for k, v in metrics.items()},
    'error_statistics': {
        'mean_abs_error': float(np.mean(abs_errors)),
        'median_abs_error': float(np.median(abs_errors)),
        'p90_abs_error': float(np.percentile(abs_errors, 90)),
        'p95_abs_error': float(np.percentile(abs_errors, 95)),
        'max_abs_error': float(np.max(abs_errors)),
    },
    'accuracy_breakdown': {
        'pct_under_10_error': float((pct_errors < 10).mean() * 100),
        'pct_under_20_error': float((pct_errors < 20).mean() * 100),
        'pct_under_30_error': float((pct_errors < 30).mean() * 100),
    },
    'test_set_size': len(test_idx),
}

with open(OUTPUT_DIR / 'evaluation_results.json', 'w') as f:
    json.dump(evaluation_results, f, indent=2)

print(f"Evaluation results saved to {OUTPUT_DIR / 'evaluation_results.json'}")

## 9. SageMaker Deployment Preparation (Optional)

In [ ]:
# Create inference script for SageMaker (commented out - uncomment to create)

# inference_script = '''
# import os
# import json
# import torch
# import pickle
# import numpy as np

# def model_fn(model_dir):
#     """Load model for SageMaker inference."""
#     checkpoint = torch.load(os.path.join(model_dir, 'best_model.pt'), map_location='cpu')
#     with open(os.path.join(model_dir, 'preprocessor.pkl'), 'rb') as f:
#         preprocessor = pickle.load(f)
#     return {'checkpoint': checkpoint, 'preprocessor': preprocessor}

# def input_fn(request_body, request_content_type):
#     """Parse input data."""
#     if request_content_type == 'application/json':
#         return json.loads(request_body)
#     raise ValueError(f"Unsupported content type: {request_content_type}")

# def predict_fn(input_data, model):
#     """Make predictions."""
#     # Process and predict here
#     pass

# def output_fn(prediction, accept):
#     """Format output."""
#     return json.dumps({'predictions': prediction.tolist()})
# '''

# with open(OUTPUT_DIR / 'inference.py', 'w') as f:
#     f.write(inference_script)
# print("SageMaker inference script created")

print("SageMaker deployment preparation ready (uncomment code above to generate inference.py)")

## 10. Predict Revenue for ALL Inactive Sites

The training data only includes **Active** sites (~26K). There are ~31K additional sites with other statuses:
- Temporarily Deactivated
- Awaiting Installation
- Deactivated
- Awaiting Reactivation
- Cancelled
- Awaiting Deactivation

We'll use the trained model to predict expected revenue for these sites if they were active.

---

## Summary

This notebook has:
1. ✅ Loaded trained model from checkpoint
2. ✅ Generated predictions on test set
3. ✅ Calculated comprehensive metrics (MAE, RMSE, SMAPE, R²)
4. ✅ Created visualizations (actual vs predicted, residuals)
5. ✅ Analyzed errors by revenue range
6. ✅ Created production-ready SiteScorer class
7. ✅ Saved evaluation results
8. ✅ Predicted revenue for ALL inactive sites (Section 10)

**Artifacts created:**
- `outputs/evaluation_plots.png` - Visualization of model performance
- `outputs/evaluation_results.json` - Comprehensive metrics
- `outputs/inactive_site_predictions.csv` - Revenue predictions for all inactive sites

**Next:** Proceed to `05_model_comparison.ipynb` for XGBoost vs Neural Network comparison, or skip to `06_explainability.ipynb` for model interpretability (SHAP, calibration, tiers).

In [ ]:
# Load the FULL dataset (all sites, not just training data)
FULL_DATA_PATH = Path('../data/processed/site_aggregated_precleaned.parquet')

if FULL_DATA_PATH.exists():
    df_full = pl.read_parquet(FULL_DATA_PATH)
    print(f"Full dataset: {len(df_full):,} sites")
    
    # Show status distribution
    status_dist = df_full.group_by('status').len().sort('len', descending=True)
    print(f"\nStatus distribution:")
    print(status_dist)
    
    # Filter to inactive sites only (NOT 'Active')
    df_inactive = df_full.filter(pl.col('status') != 'Active')
    print(f"\nInactive sites to predict: {len(df_inactive):,}")
else:
    print(f"ERROR: {FULL_DATA_PATH} not found!")
    print("Run the data pipeline first to generate this file.")

In [ ]:
# Generate predictions for all inactive sites using the SiteScorer
print("Generating revenue predictions for inactive sites...")
print("(This may take a minute for ~31K sites)")

# Use the scorer we defined earlier
inactive_predictions = scorer.predict(df_inactive)

print(f"\n✓ Generated {len(inactive_predictions):,} predictions")
print(f"\nPrediction statistics:")
print(f"  Mean predicted revenue:   ${np.mean(inactive_predictions):,.2f}")
print(f"  Median predicted revenue: ${np.median(inactive_predictions):,.2f}")
print(f"  Min predicted revenue:    ${np.min(inactive_predictions):,.2f}")
print(f"  Max predicted revenue:    ${np.max(inactive_predictions):,.2f}")

In [ ]:
# Create output DataFrame with site IDs and predictions
inactive_results = df_inactive.select([
    'gtvid',
    'id_gbase', 
    'status',
    'network',
    'retailer',
    'state',
    'active_months',
]).with_columns([
    pl.Series('predicted_avg_monthly_revenue', inactive_predictions),
])

# Sort by predicted revenue (highest first)
inactive_results = inactive_results.sort('predicted_avg_monthly_revenue', descending=True)

print("Preview of predictions (top 20 by predicted revenue):")
print("=" * 100)
print(f"{'GTVID':<10} {'Status':<25} {'Network':<12} {'State':<6} {'Months':<8} {'Predicted Revenue':>18}")
print("-" * 100)

for row in inactive_results.head(20).iter_rows(named=True):
    print(f"{row['gtvid']:<10} {row['status']:<25} {str(row['network'])[:10]:<12} {str(row['state']):<6} {row['active_months']:<8} ${row['predicted_avg_monthly_revenue']:>16,.2f}")

In [ ]:
# Analyze predictions by status
print("\nPredicted Revenue by Status:")
print("=" * 70)

status_summary = inactive_results.group_by('status').agg([
    pl.len().alias('count'),
    pl.col('predicted_avg_monthly_revenue').mean().alias('mean_predicted'),
    pl.col('predicted_avg_monthly_revenue').median().alias('median_predicted'),
    pl.col('predicted_avg_monthly_revenue').quantile(0.90).alias('p90_predicted'),
]).sort('count', descending=True)

print(f"{'Status':<25} {'Count':>8} {'Mean':>12} {'Median':>12} {'P90':>12}")
print("-" * 70)
for row in status_summary.iter_rows(named=True):
    print(f"{row['status']:<25} {row['count']:>8,} ${row['mean_predicted']:>10,.0f} ${row['median_predicted']:>10,.0f} ${row['p90_predicted']:>10,.0f}")

# High-value inactive sites (potential reactivation candidates)
high_value_threshold = np.percentile(inactive_predictions, 90)
high_value_sites = inactive_results.filter(
    pl.col('predicted_avg_monthly_revenue') >= high_value_threshold
)
print(f"\n★ High-value inactive sites (top 10%): {len(high_value_sites):,} sites")
print(f"  Predicted revenue >= ${high_value_threshold:,.0f}/month")
print(f"  These are strong candidates for reactivation efforts!")

In [ ]:
# Save predictions to CSV
output_file = OUTPUT_DIR / 'inactive_site_predictions.csv'
inactive_results.write_csv(output_file)

print(f"✓ Saved {len(inactive_results):,} predictions to: {output_file}")
print(f"  File size: {output_file.stat().st_size / 1024:.1f} KB")

# Also save high-value sites separately for easy access
high_value_file = OUTPUT_DIR / 'high_value_inactive_sites.csv'
high_value_sites.write_csv(high_value_file)
print(f"✓ Saved {len(high_value_sites):,} high-value sites to: {high_value_file}")

# Summary statistics
print(f"\n" + "=" * 60)
print("INACTIVE SITE PREDICTIONS COMPLETE")
print("=" * 60)
print(f"  Total inactive sites:     {len(inactive_results):,}")
print(f"  High-value sites (top 10%): {len(high_value_sites):,}")
print(f"  Mean predicted revenue:   ${np.mean(inactive_predictions):,.2f}")
print(f"  Total predicted revenue:  ${np.sum(inactive_predictions):,.0f}/month")
print(f"\nFiles saved:")
print(f"  - {output_file}")
print(f"  - {high_value_file}")